# 15 · Joint controlled processes

A possible future contains **every target at every horizon**. Correlated shocks, shared jump arrivals, shared hidden regimes, and evolving volatility create different kinds of dependence. This chapter builds those laws; [27 · Joint financial uncertainty](27_joint_financial_uncertainty.ipynb) makes them visible.

In [1]:
#| default_exp joint_processes

## Declare the dependence, preserve the information boundary

Scalar target parameters broadcast; vectors specify one value per target. `correlation` controls Gaussian shocks and `jump_correlation` controls jump marks. A **shared Poisson clock** triggers simultaneous vector jumps. Markov targets share one chain, inferred from all observed targets; optional `regime_correlation` matrices let dependence differ by state. The forecasting model never receives hidden states or evaluator parameters.

CCC-GARCH uses separate target variance recursions with correlated Gaussian innovations. Its conditional simulator is exact under the declared law. Multi-step cross-series covariance generally needs numerical integration; an independent Monte Carlo calculation is labelled explicitly, while means and marginal variances remain analytic. Zero correlation does not generally imply independence when jump clocks or regimes are shared.

In [2]:
#| export
from __future__ import annotations
import numpy as np
from scipy.special import logsumexp
from scipy.stats import poisson
from timegym._process_common import (
    resolve_process, _PROCESS_REGISTRY, _CHANGE_FIELDS, _TAIL_TOLERANCE,
    _finite, _exp, _factor, _emission_law, _state_parameters, _streams,
    _to_values, _observed_returns, _transform_samples, _build_case,
)


## Validate shapes and dependence matrices

In [6]:
#| export
def resolve_joint_process(kind, parameters, context_length, horizon, n_targets, seed):
    """Compatibility entry point for the shared controlled-process grammar."""
    return resolve_process(kind, parameters, context_length, horizon, n_targets, seed)


## Independent random streams for data, forecasts, and numerical moments

In [10]:
#| export
def _draw_emission(rng, n, law):
    d = len(law['mean'])
    counts = rng['jump_counts'].poisson(law['intensity'], size=n)
    return (law['drift']+rng['diffusion'].standard_normal((n, d))@law['diffusion_factor'].T
            + counts[:, None]*law['jump_mean']
            + np.sqrt(counts)[:, None]*(rng['jump_sizes'].standard_normal((n, d))@law['jump_factor'].T))

In [14]:
#| export
def _blocks_to_covariance(blocks, *, cumulative=False):
    """Blocks use (horizon, target, horizon, target), matching C-order samples."""
    if cumulative:
        blocks = blocks.cumsum(axis=0).cumsum(axis=2)
    h, d = blocks.shape[:2]
    return blocks.reshape(h*d, h*d)

## Additive components and accumulated vector shocks

In [15]:
#| export
def _joint_composite(p, length, horizon, d, seed):
    t = np.arange(length+horizon)[:, None]
    schedule = {}
    for key in _CHANGE_FIELDS:
        a = np.broadcast_to(p[key], (len(t), d)).copy()
        if p['change_point'] is not None and key in p['changes']:
            a[np.arange(len(t)) >= p['change_point']] = p['changes'][key]
        schedule[key] = a
    signal = (schedule['intercept']+schedule['slope']*t+schedule['amplitude']
              *np.sin(2*np.pi*t/schedule['period']+schedule['phase']))
    corr = np.asarray(p['correlation']); factor = _factor(corr)
    jm = np.asarray(p['jump_mean']); js = np.asarray(p['jump_sigma'])
    jump_cov = np.outer(js, js)*p['jump_correlation']; jump_factor = _factor(jump_cov)
    rng = _streams(seed, joint=True)
    def innovation(streams, n, sigma):
        shock = (streams['diffusion'].standard_normal((n, d))@factor.T)*sigma
        if p['jump_rate']:
            counts = streams['jump_counts'].poisson(p['jump_rate'], size=n)
            shock += counts[:, None]*jm+np.sqrt(counts)[:, None]*(streams['jump_sizes'].standard_normal((n, d))@jump_factor.T)
        return shock
    residual = np.empty_like(signal)
    for i in range(len(t)):
        shock = innovation(rng, 1, schedule['sigma'][i])[0]
        if p['residual'] == 'random_walk' and i == 0:
            residual[i] = 0.
        else:
            residual[i] = shock
            if p['residual'] == 'random_walk':
                residual[i] += residual[i-1]
    values = signal+residual
    last_residual = values[length-1]-signal[length-1]
    increments = np.array([np.outer(s, s)*corr+p['jump_rate']*(jump_cov+np.outer(jm, jm))
                           for s in schedule['sigma'][length:]])
    mean = signal[length:].copy()
    blocks = np.zeros((horizon, d, horizon, d))
    for h in range(horizon):
        blocks[h, :, h, :] = increments[h]
    cumulative = p['residual'] == 'random_walk'
    if cumulative:
        mean += last_residual+np.arange(1, horizon+1)[:, None]*p['jump_rate']*jm
    def sampler(n, sample_seed):
        streams = _streams(sample_seed, role=1, joint=True)
        out = np.empty((n, horizon, d)); previous = np.broadcast_to(last_residual, (n, d)).copy()
        for h in range(horizon):
            shocks = innovation(streams, n, schedule['sigma'][length+h])
            previous = previous+shocks if cumulative else shocks
            out[:, h] = signal[length+h]+previous
        return out
    info = dict(residual=p['residual'], future_parameter_schedule_known=True,
                schedule_convention='absolute_time_intercept_plus_slope_t',
                residual_level_from_observed_history=last_residual.tolist() if cumulative else None,
                dependence='correlated_innovations_and_shared_Poisson_jump_clock')
    return values, mean, _blocks_to_covariance(blocks, cumulative=cumulative), info, sampler if p['jump_rate'] else None, {}

## Joint Merton moments in return, log-price, and price units

In [16]:
#| export
def _joint_merton(p, length, horizon, d, seed):
    law = _emission_law(p); rng = _streams(seed, joint=True)
    count = length+horizon-(p['output'] != 'log_return')
    returns = np.array([_draw_emission(rng, 1, law)[0] for _ in range(count)])
    values = _to_values(returns, p); last = values[length-1]
    blocks = np.zeros((horizon, d, horizon, d))
    mean = np.broadcast_to(law['mean'], (horizon, d)).copy()
    for h in range(horizon):
        blocks[h, :, h, :] = law['covariance']
    if p['output'] == 'price':
        mean = _exp(np.log(last)+np.arange(1, horizon+1)[:, None]*np.array(p['mu'])*p['dt'], 'Joint price mean')
        for h in range(horizon):
            for j in range(horizon):
                blocks[h, :, j, :] = np.outer(mean[h], mean[j])*np.expm1(min(h+1, j+1)*law['spread'])
    elif p['output'] == 'log_price':
        mean = last+mean.cumsum(axis=0)
    covariance = _blocks_to_covariance(blocks, cumulative=p['output'] == 'log_price')
    def sampler(n, sample_seed):
        streams = _streams(sample_seed, role=1, joint=True)
        draws = np.stack([_draw_emission(streams, n, law) for _ in range(horizon)], axis=1)
        return _transform_samples(draws, p, last)
    info = dict(output_units=p['output'], drift_convention='physical_expected_proportional_price_growth',
                dependence='correlated_diffusion_and_shared_Poisson_jump_clock')
    gaussian = p['jump_rate'] == 0 and p['output'] != 'price'
    return values, mean, covariance, info, None if gaussian else sampler, {}

## CCC-GARCH: exact paths, declared numerical cross-moments

In [17]:
#| export
def _joint_garch(p, length, horizon, d, seed):
    mu, omega, alpha, beta, initial = [np.asarray(p[key]) for key in ('mu', 'omega', 'alpha', 'beta', 'initial_variance')]
    corr = np.asarray(p['correlation']); factor = _factor(corr)
    count = length+horizon-(p['output'] != 'log_return')
    rng = _streams(seed, joint=True)['diffusion']; variance = initial.copy()
    returns = np.empty((count, d))
    for t in range(count):
        shock = np.sqrt(variance)*(rng.standard_normal(d)@factor.T)
        returns[t] = mu+shock
        variance = omega+alpha*shock**2+beta*variance
    values = _to_values(returns, p); last = values[length-1]
    next_variance = initial.copy()
    for value in _observed_returns(values, p, length):
        next_variance = omega+alpha*(value-mu)**2+beta*next_variance
    expected = np.empty((horizon, d)); expected[0] = next_variance
    for h in range(1, horizon):
        expected[h] = omega+(alpha+beta)*expected[h-1]
    def simulate(n, sample_seed, role=1, moments=False):
        streams = _streams(sample_seed, role=role, joint=True); rng = streams['diffusion']
        v = np.broadcast_to(next_variance, (n, d)).copy()
        out = np.empty((n, horizon, d)) if not moments else np.empty((horizon, d, d))
        for h in range(horizon):
            roots = np.sqrt(v)
            if moments:
                # Average conditional covariance, not noisy sample return covariance.
                q = (roots.T@roots/n)*corr
                correction = np.sqrt(expected[h]/np.diag(q))
                out[h] = q*np.outer(correction, correction)
                np.fill_diagonal(out[h], expected[h])
            shock = roots*(rng.standard_normal((n, d))@factor.T)
            if not moments:
                out[:, h] = mu+shock
            v = omega+alpha*shock**2+beta*v
        return out
    # The first-step covariance, diagonal-correlation case and deterministic
    # variance paths are analytic. Otherwise E[sqrt(v_i v_j)] needs integration.
    numerical = horizon > 1 and np.any(alpha > 0) and np.any(corr != np.eye(d))
    if numerical:
        q = simulate(p['moment_samples'], p['moment_seed'], role=2, moments=True)
    else:
        q = np.array([np.outer(np.sqrt(v), np.sqrt(v))*corr for v in expected])
    blocks = np.zeros((horizon, d, horizon, d))
    for h in range(horizon):
        blocks[h, :, h, :] = q[h]
    mean = np.broadcast_to(mu, (horizon, d)).copy()
    if p['output'] == 'log_price':
        mean = last+mean.cumsum(axis=0)
    def sampler(n, sample_seed):
        return _transform_samples(simulate(n, sample_seed), p, last)
    info = dict(output_units=p['output'], dependence='constant_conditional_correlation_GARCH',
                next_variance_from_observed_history=next_variance.tolist(),
                moment_random_stream='independent_of_observations_and_reference_samples')
    capabilities = dict(moments_kind='analytic_mean_monte_carlo_covariance',
                        moment_samples=p['moment_samples'], moment_seed=p['moment_seed']) if numerical else {}
    return values, mean, _blocks_to_covariance(blocks, cumulative=p['output'] == 'log_price'), info, sampler, capabilities

## Filter one hidden chain using every observed target

In [19]:
#| export
def _joint_filter(returns, p, laws):
    transition = np.asarray(p['transition']); posterior = np.asarray(p['initial_probs'])
    components, tails = [], []
    for law in laws:
        rate = law['intensity']
        limit = 0 if rate == 0 else int(poisson.ppf(1-_TAIL_TOLERANCE, rate))
        counts = np.arange(limit+1)
        centers = law['drift']+counts[:, None]*law['jump_mean']
        diffusion = law['diffusion_factor']@law['diffusion_factor'].T
        jump = law['jump_factor']@law['jump_factor'].T
        covariance = diffusion[None, :, :]+counts[:, None, None]*jump
        factors = np.linalg.cholesky(covariance)
        inverse = np.linalg.inv(factors)
        logdet = 2*np.log(np.diagonal(factors, axis1=1, axis2=2)).sum(axis=1)
        constants = poisson.logpmf(counts, rate)-.5*(returns.shape[1]*np.log(2*np.pi)+logdet)
        components.append((centers, inverse, constants))
        tails.append(float(poisson.sf(limit, rate)))
    for value in returns:
        prior = posterior@transition
        likelihood = []
        for centers, inverse, constants in components:
            whitened = np.einsum('nij,nj->ni', inverse, value-centers)
            likelihood.append(logsumexp(constants-.5*np.sum(whitened**2, axis=1)))
        with np.errstate(divide='ignore'):
            logweights = np.log(prior)+likelihood
        normalizer = logsumexp(logweights)
        if not np.isfinite(normalizer):
            raise ValueError('Observed vector has zero numerical likelihood under the regime filter')
        posterior = np.exp(logweights-normalizer)
    return posterior, max(tails)

## Propagate uncertainty over states, horizons, and targets

In [20]:
#| export
def _joint_markov_moments(posterior, transition, laws, horizon, last, output):
    k, d = len(laws), len(last)
    blocks = np.empty((horizon, d, horizon, d)); mean = np.empty((horizon, d))
    if output != 'price':
        a = np.array([law['mean'] for law in laws]); v = np.array([law['covariance'] for law in laws])
        powers = [np.linalg.matrix_power(transition, h) for h in range(horizon+1)]
        probs = [posterior@powers[h+1] for h in range(horizon)]
        for h in range(horizon):
            mean[h] = a[0]+probs[h]@(a-a[0])
        for h in range(horizon):
            centered = a-mean[h]
            blocks[h, :, h, :] = np.einsum('k,kij->ij', probs[h], v)+centered.T@(probs[h][:, None]*centered)
            for j in range(h+1, horizon):
                future_centered = a[0]+powers[j-h]@(a-a[0])-mean[j]
                cross = centered.T@(probs[h][:, None]*future_centered)
                blocks[h, :, j, :] = cross; blocks[j, :, h, :] = cross.T
        if output == 'log_price':
            mean = last+mean.cumsum(axis=0)
        return mean, _blocks_to_covariance(blocks, cumulative=output == 'log_price')
    growth = np.array([law['growth'] for law in laws]); gcov = np.array([law['growth_covariance'] for law in laws])
    gsecond = growth[:, :, None]*growth[:, None, :]+gcov
    probs = posterior.copy(); state_mean = np.broadcast_to(last, (k, d)).copy()
    state_cov = np.zeros((k, d, d)); states = []
    for h in range(horizon):
        joint = probs[:, None]*transition; next_probs = joint.sum(axis=0)
        weights = np.divide(joint, next_probs[None, :], out=np.zeros_like(joint), where=next_probs[None, :] > 0)
        incoming_mean = state_mean[0]+weights.T@(state_mean-state_mean[0])
        delta = state_mean[:, None, :]-incoming_mean[None, :, :]
        incoming_cov = np.einsum('ab,aij->bij', weights, state_cov)+np.einsum('ab,abi,abj->bij', weights, delta, delta)
        state_cov = incoming_cov*gsecond+(incoming_mean[:, :, None]*incoming_mean[:, None, :])*gcov
        state_mean = incoming_mean*growth; probs = next_probs/next_probs.sum()
        mean[h] = state_mean[0]+probs@(state_mean-state_mean[0])
        states.append((probs.copy(), state_mean.copy(), state_cov.copy()))
    for h, (probs, state_mean, state_cov) in enumerate(states):
        future_growth = np.ones((k, d)); centered = state_mean-mean[h]
        for j in range(h, horizon):
            if j > h:
                future_growth = transition@(growth*future_growth)
            cross = np.einsum('k,kij,kj->ij', probs, state_cov, future_growth)
            cross += centered.T@(probs[:, None]*(state_mean*future_growth-mean[j]))
            blocks[h, :, j, :] = cross; blocks[j, :, h, :] = cross.T
    return mean, _blocks_to_covariance(blocks)

In [21]:
#| export
def _joint_markov(p, length, horizon, d, seed):
    transition = np.asarray(p['transition']); k = len(transition)
    laws = [_emission_law(_state_parameters(p, s)) for s in range(k)]
    rng = _streams(seed, joint=True); state = rng['states'].choice(k, p=p['initial_probs'])
    count = length+horizon-(p['output'] != 'log_return'); returns = np.empty((count, d))
    for t in range(count):
        state = rng['states'].choice(k, p=transition[state])
        returns[t] = _draw_emission(rng, 1, laws[state])[0]
    values = _to_values(returns, p); last = values[length-1]
    posterior, tail = _joint_filter(_observed_returns(values, p, length), p, laws)
    mean, covariance = _joint_markov_moments(posterior, transition, laws, horizon, last, p['output'])
    def sampler(n, sample_seed):
        streams = _streams(sample_seed, role=1, joint=True)
        states = streams['states'].choice(k, size=n, p=posterior)
        out = np.empty((n, horizon, d))
        for h in range(horizon):
            cumulative = transition[states].cumsum(axis=1); cumulative[:, -1] = 1.
            states = (streams['states'].random(n)[:, None] > cumulative).sum(axis=1)
            for s, law in enumerate(laws):
                selected = states == s
                if np.any(selected):
                    out[selected, h] = _draw_emission(streams, int(selected.sum()), law)
        return _transform_samples(out, p, last)
    jumps = any(p['regime_jump_rate'])
    info = dict(output_units=p['output'], state_posterior=posterior.tolist(),
                state_inference='forward_filter_all_observed_targets', hidden_states_exposed_to_model=False,
                dependence='shared_hidden_chain_correlated_diffusion_and_shared_jump_clock',
                emission_poisson_tail_bound=tail,
                filter_note='truncated_joint_emission_mixture; tail_bound_is_not_a_posterior_error_bound' if jumps else 'multivariate_Gaussian_emissions',
                moments_conditioning='numerically_filtered_posterior' if jumps else 'filtered_posterior')
    return values, mean, covariance, info, sampler, dict(sampler_kind='approximate_conditional') if jumps else {}

## Build a replayable case

In [22]:
#| export
def generate_joint_process(kind, parameters, context_length, horizon, n_targets, seed):
    """Build a joint case from parameters already returned by resolve_process."""
    builder = globals()[_PROCESS_REGISTRY[kind]['joint']]
    return _build_case(builder, kind, parameters, context_length, horizon, n_targets, seed)


## Inspect the joint contract

Each sample keeps its two targets paired. The covariance matrix uses horizon-major order: `(h1,y0), (h1,y1), (h2,y0), ...`. Quantiles are marginal intervals, not simultaneous regions.

In [23]:
from timegym import Scenario
case = Scenario('merton_jump_diffusion', n_targets=2, horizon=4, parameters={
    'correlation': .6, 'jump_correlation': .8, 'jump_mean': [-.08, -.15],
    'jump_sigma': [.08, .12], 'output': 'log_return'}).generate(seed=23)
draws = case.reference.sample(128, seed=91)
assert draws.shape == (128, 4, 2)
assert case.reference.covariance.shape == (8, 8)
case.reference.conditioning

{'observed_through': 119,
 'known_generator_parameters': True,
 'future_observations_used': False,
 'output_units': 'log_return',
 'drift_convention': 'physical_expected_proportional_price_growth',
 'dependence': 'correlated_diffusion_and_shared_Poisson_jump_clock',
 'moments_kind': 'analytic',
 'sampler_kind': 'exact_conditional',
 'quantiles_kind': 'monte_carlo',
 'quantile_samples': 8192,
 'quantile_seed': 0,
 'quantile_method': 'linear'}